<a href="https://colab.research.google.com/github/AbdallahHMK/Ecoload/blob/main/Energy_Performance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Create a Prediction Function

To make predictions for new building configurations, we'll create a function that takes the 8 input features, scales them, and then uses our trained Neural Network model to predict the Heating Load (Y1) and Cooling Load (Y2).

In [ ]:
import numpy as np

def predict_efficiency(relative_compactness, surface_area, wall_area, roof_area, overall_height, orientation, glazing_area, glazing_area_distribution):
    """
    Predicts Heating Load (Y1) and Cooling Load (Y2) for a building based on its characteristics.

    Args:
        relative_compactness (float): X1
        surface_area (float): X2
        wall_area (float): X3
        roof_area (float): X4
        overall_height (float): X5
        orientation (int): X6
        glazing_area (float): X7
        glazing_area_distribution (int): X8

    Returns:
        tuple: A tuple containing predicted Y1 (Heating Load) and Y2 (Cooling Load).
    """
    # Create a numpy array from the input values, ensuring the correct order
    input_data = np.array([[relative_compactness, surface_area, wall_area, roof_area,
                            overall_height, orientation, glazing_area, glazing_area_distribution]])

    # Scale the input data using the fitted scaler
    scaled_input_data = scaler.transform(input_data)

    # Make predictions using the trained neural network model
    predictions = model.predict(scaled_input_data)

    # Extract Y1 and Y2 predictions
    predicted_y1 = predictions[0, 0]
    predicted_y2 = predictions[0, 1]

    return predicted_y1, predicted_y2

print("Prediction function defined successfully.")

### Example Usage

Now, let's use the `predict_efficiency` function with some example building characteristics. For instance, we can use the characteristics of the first entry in our original dataset to see how well it predicts.

In [ ]:
# Example: Predict efficiency for a new building
# Using the first row of the original 'X' data for demonstration
example_building_features = X.iloc[0]

# Call the prediction function
predicted_y1_example, predicted_y2_example = predict_efficiency(
    example_building_features['X1'], example_building_features['X2'],
    example_building_features['X3'], example_building_features['X4'],
    example_building_features['X5'], example_building_features['X6'],
    example_building_features['X7'], example_building_features['X8']
)

print(f"Example Building Characteristics: \n{example_building_features.to_string()}")
print(f"\nPredicted Heating Load (Y1): {predicted_y1_example:.2f}")
print(f"Predicted Cooling Load (Y2): {predicted_y2_example:.2f}")

# You can compare these to the actual values from y.iloc[0]
actual_y1_example = y.iloc[0]['Y1']
actual_y2_example = y.iloc[0]['Y2']

print(f"Actual Heating Load (Y1): {actual_y1_example:.2f}")
print(f"Actual Cooling Load (Y2): {actual_y2_example:.2f}")

You can now call the `predict_efficiency` function with any new set of 8 building characteristics to get the predicted heating and cooling loads!

### Interactive Prediction Form

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Define the ranges and steps for each input feature based on general knowledge or dataset characteristics
# For simplicity, these ranges are illustrative and can be refined based on actual data distribution

x1_slider = widgets.FloatSlider(
    value=0.7, min=0.6, max=1.0, step=0.01,
    description='Relative Compactness (X1):',
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.2f',
)
x2_slider = widgets.FloatSlider(
    value=700, min=500, max=850, step=0.5,
    description='Surface Area (X2):',
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.1f',
)
x3_slider = widgets.FloatSlider(
    value=300, min=250, max=450, step=0.5,
    description='Wall Area (X3):',
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.1f',
)
x4_slider = widgets.FloatSlider(
    value=150, min=100, max=250, step=0.25,
    description='Roof Area (X4):',
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.2f',
)
x5_slider = widgets.FloatSlider(
    value=4.0, min=3.5, max=7.0, step=0.01,
    description='Overall Height (X5):',
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.2f',
)
x6_dropdown = widgets.Dropdown(
    options=[(str(i), i) for i in range(2, 6)],  # Orientation values 2,3,4,5
    value=3,
    description='Orientation (X6):',
)
x7_slider = widgets.FloatSlider(
    value=0.25, min=0.0, max=0.4, step=0.05,
    description='Glazing Area (X7):',
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.2f',
)
x8_dropdown = widgets.Dropdown(
    options=[(str(i), i) for i in range(0, 6)],  # Glazing Area Distribution values 0,1,2,3,4,5
    value=3,
    description='Glazing Area Dist. (X8):',
)

predict_button = widgets.Button(
    description='Predict Efficiency',
    disabled=False,
    button_style='success', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Click to predict building efficiency',
    icon='check'
)

output_widget = widgets.Output()

def on_predict_button_clicked(b):
    with output_widget:
        output_widget.clear_output()
        predicted_y1, predicted_y2 = predict_efficiency(
            x1_slider.value, x2_slider.value, x3_slider.value, x4_slider.value,
            x5_slider.value, x6_dropdown.value, x7_slider.value, x8_dropdown.value
        )
        print(f"Predicted Heating Load (Y1): {predicted_y1:.2f}")
        print(f"Predicted Cooling Load (Y2): {predicted_y2:.2f}")

predict_button.on_click(on_predict_button_clicked)

# Arrange widgets in a VBox
input_widgets = widgets.VBox([
    x1_slider, x2_slider, x3_slider, x4_slider, x5_slider,
    x6_dropdown, x7_slider, x8_dropdown, predict_button
])

display(input_widgets, output_widget)
print("Interactive form created successfully!")

In [ ]:
import joblib

# Save the trained Keras model in the recommended native Keras format
model.save('building_efficiency_model.keras')
print("Neural Network model saved as 'building_efficiency_model.keras'")

# Save the StandardScaler
joblib.dump(scaler, 'feature_scaler.pkl')
print("Scaler saved as 'feature_scaler.pkl'")

You can now download `building_efficiency_model.h5` and `feature_scaler.pkl` from your Colab files to use them in other environments.

### Streamlit Script for Model Deployment

To create a standalone Streamlit application, save the following code into a file named `app.py` in the same directory where you saved `building_efficiency_model.keras` and `feature_scaler.pkl`.

After saving, open your terminal, navigate to that directory, and run the command: `streamlit run app.py`

In [ ]:
!pip install streamlit

In [ ]:
import streamlit as st
import tensorflow as tf
import joblib
import numpy as np

# Load the trained model and scaler
@st.cache_resource
def load_resources():
    model = tf.keras.models.load_model('building_efficiency_model.keras')
    scaler = joblib.load('feature_scaler.pkl')
    return model, scaler

model, scaler = load_resources()

st.title('Building Energy Efficiency Predictor')
st.write('Predict Heating Load (Y1) and Cooling Load (Y2) based on building characteristics.')

# Input features using Streamlit widgets
st.sidebar.header('Input Building Characteristics')

x1 = st.sidebar.slider('Relative Compactness (X1)', min_value=0.60, max_value=1.00, value=0.70, step=0.01)
x2 = st.sidebar.slider('Surface Area (X2)', min_value=500.0, max_value=850.0, value=700.0, step=0.5)
x3 = st.sidebar.slider('Wall Area (X3)', min_value=250.0, max_value=450.0, value=300.0, step=0.5)
x4 = st.sidebar.slider('Roof Area (X4)', min_value=100.00, max_value=250.00, value=150.00, step=0.25)
x5 = st.sidebar.slider('Overall Height (X5)', min_value=3.50, max_value=7.00, value=4.00, step=0.01)
x6 = st.sidebar.selectbox('Orientation (X6)', options=[2, 3, 4, 5], index=1) # Default to 3
x7 = st.sidebar.slider('Glazing Area (X7)', min_value=0.00, max_value=0.40, value=0.25, step=0.05)
x8 = st.sidebar.selectbox('Glazing Area Distribution (X8)', options=[0, 1, 2, 3, 4, 5], index=3) # Default to 3

if st.sidebar.button('Predict Efficiency'):
    # Prepare input data for prediction
    input_data = np.array([[x1, x2, x3, x4, x5, x6, x7, x8]])

    # Scale the input data
    scaled_input_data = scaler.transform(input_data)

    # Make predictions
    predictions = model.predict(scaled_input_data)
    predicted_y1 = predictions[0, 0]
    predicted_y2 = predictions[0, 1]

    st.subheader('Prediction Results:')
    st.write(f'**Predicted Heating Load (Y1):** {predicted_y1:.2f}')
    st.write(f'**Predicted Cooling Load (Y2):** {predicted_y2:.2f}')

# Optional: Display raw inputs
st.sidebar.write('---')
st.sidebar.write('Current Inputs:')
st.sidebar.write(f'X1: {x1}')
st.sidebar.write(f'X2: {x2}')
st.sidebar.write(f'X3: {x3}')
st.sidebar.write(f'X4: {x4}')
st.sidebar.write(f'X5: {x5}')
st.sidebar.write(f'X6: {x6}')
st.sidebar.write(f'X7: {x7}')
st.sidebar.write(f'X8: {x8}')

In [ ]:
# Install pyngrok to easily manage ngrok tunnels
!pip install pyngrok -q

from pyngrok import ngrok
import os

# Terminate open ngrok tunnels to ensure a clean start
ngrok.kill()

# Get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
# If you haven't added it to your ngrok config, uncomment the line below and replace 'YOUR_AUTHTOKEN'
# ngrok_auth_token = os.environ.get('NGROK_AUTHTOKEN') # You can also set this as an environment variable
# if ngrok_auth_token is None:
#     # For demonstration, hardcode or ask user to paste if not set via env var
#     print("Please enter your ngrok authtoken. Get it from https://dashboard.ngrok.com/get-started/your-authtoken")
#     ngrok_auth_token = input()
#     ngrok.set_auth_token(ngrok_auth_token)

# If you have already set your authtoken previously or using a persistent environment,
# pyngrok might pick it up automatically.
# Otherwise, you can set it directly if not using environment variables:
ngrok.set_auth_token("3CK34UdnB1NjUzPbch84fLs7jup_6ett8fuZaKArmoWAvdww5") # Replace with your actual authtoken

# Open a ngrok tunnel to the Streamlit app's port (8501)
# The Streamlit app was started in the background in a previous cell.
try:
    public_url = ngrok.connect(8501)
    print(f"Streamlit app available at: {public_url}")
except Exception as e:
    print(f"Error creating ngrok tunnel: {e}")
    print("Please ensure your ngrok authtoken is correctly set. You can set it using: `ngrok.set_auth_token('YOUR_AUTHTOKEN')`")
    print("Or set it via environment variable 'NGROK_AUTHTOKEN'.")

In [ ]:
# Ensure any previous Streamlit processes are stopped
!kill -9 $(lsof -t -i:8501) 2>/dev/null

import time
import os
from pyngrok import ngrok

print("Restarting Streamlit app...")
# Start the Streamlit app in the background
!streamlit run app.py &>/dev/null&

# Give Streamlit a moment to start up
time.sleep(5)
print("Streamlit app restarted. Attempting to re-establish ngrok tunnel...")

# Terminate existing ngrok tunnels
ngrok.kill()

# Make sure your authtoken is set. If not, uncomment the line below and replace with your token.
ngrok.set_auth_token("3CK34UdnB1NjUzPbch84fLs7jup_6ett8fuZaKArmoWAvdww5")

try:
    public_url = ngrok.connect(8501)
    print(f"Streamlit app available at: {public_url}")
    print("If the link still doesn't work, please try refreshing the page or restarting your Colab runtime.")
except Exception as e:
    print(f"Error creating ngrok tunnel: {e}")
    print("Please ensure your ngrok authtoken is correctly set and try again. Sometimes a Colab runtime restart helps.")

In [ ]:
!streamlit run app.py &>/dev/null&

In [ ]:
!npm install -g localtunnel

Now you have an interactive form above. Adjust the sliders and dropdowns to change the building characteristics, then click 'Predict Efficiency' to see the predicted Heating and Cooling Loads based on the Neural Network model.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a scatter plot for Y1 (Heating Load) using NN predictions
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test['Y1'], y=y_pred_nn[:, 0])
plt.plot([y_test['Y1'].min(), y_test['Y1'].max()], [y_test['Y1'].min(), y_test['Y1'].max()], 'r--') # Add a diagonal line for perfect prediction
plt.title('NN: Actual vs. Predicted Y1 (Heating Load)')
plt.xlabel('Actual Y1')
plt.ylabel('Predicted Y1')
plt.grid(True)
plt.show()

In [ ]:
# Create a scatter plot for Y2 (Cooling Load) using NN predictions
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test['Y2'], y=y_pred_nn[:, 1])
plt.plot([y_test['Y2'].min(), y_test['Y2'].max()], [y_test['Y2'].min(), y_test['Y2'].max()], 'r--') # Add a diagonal line for perfect prediction
plt.title('NN: Actual vs. Predicted Y2 (Cooling Load)')
plt.xlabel('Actual Y2')
plt.ylabel('Predicted Y2')
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a scatter plot for Y1 (Heating Load) using NN predictions
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test['Y1'], y=y_pred_nn[:, 0])
plt.plot([y_test['Y1'].min(), y_test['Y1'].max()], [y_test['Y1'].min(), y_test['Y1'].max()], 'r--') # Add a diagonal line for perfect prediction
plt.title('NN: Actual vs. Predicted Y1 (Heating Load)')
plt.xlabel('Actual Y1')
plt.ylabel('Predicted Y1')
plt.grid(True)
plt.show()

In [ ]:
# Create a scatter plot for Y2 (Cooling Load) using NN predictions
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test['Y2'], y=y_pred_nn[:, 1])
plt.plot([y_test['Y2'].min(), y_test['Y2'].max()], [y_test['Y2'].min(), y_test['Y2'].max()], 'r--') # Add a diagonal line for perfect prediction
plt.title('NN: Actual vs. Predicted Y2 (Cooling Load)')
plt.xlabel('Actual Y2')
plt.ylabel('Predicted Y2')
plt.grid(True)
plt.show()

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Define the neural network model
model = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),  # Input layer matches number of features
    layers.Dense(64, activation='relu'),              # Hidden layer with 64 neurons and ReLU activation
    layers.Dense(64, activation='relu'),              # Another hidden layer with 64 neurons and ReLU activation
    layers.Dense(y_train.shape[1])                    # Output layer with 2 neurons (for Y1 and Y2)
])

# Compile the model
model.compile(optimizer='adam', loss='mse', metrics=['mae', 'mse'])

model.summary()

In [ ]:
# Train the model
history = model.fit(
    X_train_scaled, y_train,
    epochs=100,             # Number of training epochs
    batch_size=32,          # Batch size for training
    validation_split=0.2,   # Use 20% of training data for validation
    verbose=0               # Suppress verbose output during training
)

print("Neural Network model trained successfully.")

In [ ]:
# Evaluate the model on the test data
loss, mae, mse = model.evaluate(X_test_scaled, y_test, verbose=0)

print(f"Neural Network Test Loss: {loss:.2f}")
print(f"Neural Network Test Mean Absolute Error (MAE): {mae:.2f}")
print(f"Neural Network Test Mean Squared Error (MSE): {mse:.2f}")

# Make predictions with the neural network
y_pred_nn = model.predict(X_test_scaled)

# Calculate R-squared for NN predictions
from sklearn.metrics import r2_score
r2_y1_nn = r2_score(y_test['Y1'], y_pred_nn[:, 0])
r2_y2_nn = r2_score(y_test['Y2'], y_pred_nn[:, 1])

print(f"Neural Network R-squared for Y1 (Heating Load): {r2_y1_nn:.2f}")
print(f"Neural Network R-squared for Y2 (Cooling Load): {r2_y2_nn:.2f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a scatter plot for Y1 (Heating Load)
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test['Y1'], y=y_pred[:, 0])
plt.plot([y_test['Y1'].min(), y_test['Y1'].max()], [y_test['Y1'].min(), y_test['Y1'].max()], 'r--') # Add a diagonal line for perfect prediction
plt.title('Actual vs. Predicted Y1 (Heating Load)')
plt.xlabel('Actual Y1')
plt.ylabel('Predicted Y1')
plt.grid(True)
plt.show()

In [ ]:
# Create a scatter plot for Y2 (Cooling Load)
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test['Y2'], y=y_pred[:, 1])
plt.plot([y_test['Y2'].min(), y_test['Y2'].max()], [y_test['Y2'].min(), y_test['Y2'].max()], 'r--') # Add a diagonal line for perfect prediction
plt.title('Actual vs. Predicted Y2 (Cooling Load)')
plt.xlabel('Actual Y2')
plt.ylabel('Predicted Y2')
plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import r2_score

# Calculate R-squared for Y1 (Heating Load)
r2_y1 = r2_score(y_test['Y1'], y_pred[:, 0])
print(f"R-squared for Y1 (Heating Load): {r2_y1:.2f}")

# Calculate R-squared for Y2 (Cooling Load)
r2_y2 = r2_score(y_test['Y2'], y_pred[:, 1])
print(f"R-squared for Y2 (Cooling Load): {r2_y2:.2f}")

In [ ]:
from sklearn.metrics import mean_squared_error

# Make predictions on the scaled test data
y_pred = linear_model.predict(X_test_scaled)

# Calculate Mean Squared Error for Y1 (Heating Load)
mse_y1 = mean_squared_error(y_test['Y1'], y_pred[:, 0])
print(f"Mean Squared Error for Y1 (Heating Load): {mse_y1:.2f}")

# Calculate Mean Squared Error for Y2 (Cooling Load)
mse_y2 = mean_squared_error(y_test['Y2'], y_pred[:, 1])
print(f"Mean Squared Error for Y2 (Cooling Load): {mse_y2:.2f}")


In [ ]:
from sklearn.linear_model import LinearRegression

# Initialize the Linear Regression model
linear_model = LinearRegression()

# Train the model using the scaled training data
linear_model.fit(X_train_scaled, y_train)

print("Linear Regression model trained successfully.")
print(f"Coefficients: {linear_model.coef_}")
print(f"Intercept: {linear_model.intercept_}")

In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize the StandardScaler
scaler = StandardScaler()

# Fit the scaler on the training features (X_train) and transform them
X_train_scaled = scaler.fit_transform(X_train)

# Transform the test features (X_test) using the fitted scaler
X_test_scaled = scaler.transform(X_test)

print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape: {X_test_scaled.shape}")


In [ ]:
from sklearn.model_selection import train_test_split

# Define features (X) and target variables (y)
X = df[['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8']]
y = df[['Y1', 'Y2']]

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# Display descriptive statistics for the numerical columns
print("\nDescriptive Statistics:")
display(df.describe())

In [ ]:
import pandas as pd

# Load the dataset from the CSV file
df = pd.read_csv('/content/ENB2012_data.csv')

# Display the first 5 rows of the DataFrame
print("First 5 rows of the dataset:")
display(df.head())

In [ ]:
# Display information about the DataFrame to check data types and non-null values
print("\nDataFrame Info:")
display(df.info())